# Feature Engineering

**OJO: Al finalizar este notebook van a ver, varias clases a predecir llamadas "clase_...". Cuando se seleccione para entrenar se debe seleccionar una y descartar las otras. No incluirlas en las features de entrenamiento!!**

In [59]:
# Importar dataset
import polars as pl
df = pl.read_csv("sell-in.txt.gz", separator="\t")
print(df)

shape: (2_945_818, 7)
┌─────────┬─────────────┬────────────┬─────────────────┬────────────────┬────────────────┬─────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_precios_cu ┆ cust_request_q ┆ cust_request_t ┆ tn      │
│ ---     ┆ ---         ┆ ---        ┆ idados          ┆ ty             ┆ n              ┆ ---     │
│ i64     ┆ i64         ┆ i64        ┆ ---             ┆ ---            ┆ ---            ┆ f64     │
│         ┆             ┆            ┆ i64             ┆ i64            ┆ f64            ┆         │
╞═════════╪═════════════╪════════════╪═════════════════╪════════════════╪════════════════╪═════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0               ┆ 2              ┆ 0.053          ┆ 0.053   │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.13628        ┆ 0.13628 │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.03028        ┆ 0.03028 │
│ 201701  ┆ 10125       ┆ 20524      ┆ 0               ┆ 1           

## Definición de granularidad (Producto/Producto-Cliente)

In [60]:
#granularidad:
 # p: producto
 # pc: producto-cliente
granularidad = "pc"   


if granularidad == "p":
    # Por cada producto, se evalua cual es el período más antiguo en el que se vendió este producto para algún cliente. Luego se agrega este dato en una columna llamado "first_sell_in_period" manteniendo el dataset original. Para esto se puede usar la función groupby y agg de polars.
    df = df.with_columns(
        pl.col("periodo").min().over("product_id").alias("first_sell_in_period")
    )  
    print(df)
elif granularidad == "pc":
    df = df.with_columns(
    pl.col("periodo").min().over(["product_id", "customer_id"]).alias("first_sell_in_period")
    )  
    print(df)
else:
    print("Granularidad no válida. Por favor, elija 'p' o 'pc'.")



shape: (2_945_818, 8)
┌─────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn      ┆ first_sell │
│ ---     ┆ ---         ┆ ---        ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---     ┆ _in_period │
│ i64     ┆ i64         ┆ i64        ┆ s          ┆ ---        ┆ ---        ┆ f64     ┆ ---        │
│         ┆             ┆            ┆ ---        ┆ i64        ┆ f64        ┆         ┆ i64        │
│         ┆             ┆            ┆ i64        ┆            ┆            ┆         ┆            │
╞═════════╪═════════════╪════════════╪════════════╪════════════╪════════════╪═════════╪════════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0          ┆ 2          ┆ 0.053      ┆ 0.053   ┆ 201701     │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.13628    ┆ 0.13628 ┆ 201701     │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.03

## Completado de 0s y Nulls

In [61]:
# Para cada combinación de producto y cliente se completan los períodos faltantes entre el período más antiguo (first_sell_in_period) y el período más reciente (201912).
# Hay que completar la columna periodo con el periodo faltante correspondiente en el formato AAAAMM.
# Las columnas customer_id, product_id se mantienen constantes para cada combinación de producto y cliente.  
# Las columnas plan_precios_cuidados y first_sell_in_period se completan con el valor correspondiente a cada producto
# Las columnas cust_request_qty, cust_request_tn, tn se completan con el valor 0.

# Definimos el límite superior de los periodos como entero
FECHA_LIMITE = 201912

# 1. Creamos la grilla con todos los periodos teóricos en formato AAAAMM
grilla_base = (
    df.select(["customer_id", "product_id", "first_sell_in_period"])
    .unique()
    .filter(pl.col("first_sell_in_period") <= FECHA_LIMITE)
    .with_columns(
        # Convertimos AAAAMM a una escala lineal de meses: (Año * 12) + Mes
        inicio_meses=(pl.col("first_sell_in_period") // 100) * 12 + (pl.col("first_sell_in_period") % 100),
        fin_meses=(FECHA_LIMITE // 100) * 12 + (FECHA_LIMITE % 100)
    )
    .with_columns(
        # Generamos el rango de pasos de meses a rellenar
        pasos=pl.int_ranges(0, pl.col("fin_meses") - pl.col("inicio_meses") + 1)
    )
    .explode("pasos")
)

# Reconstruimos la columna 'periodo' en formato entero AAAAMM
grilla = grilla_base.with_columns(
    mes_actual_lineal=pl.col("inicio_meses") + pl.col("pasos")
).with_columns(
    periodo=(
        ((pl.col("mes_actual_lineal") - 1) // 12) * 100 + 
        ((pl.col("mes_actual_lineal") - 1) % 12 + 1)
    ).cast(pl.Int64)
).select(["customer_id", "product_id", "periodo"])

# 2. Hacemos el Left Join con tus datos originales
df_completado = grilla.join(df, on=["customer_id", "product_id", "periodo"], how="left")

# 3. Rellenamos las columnas según tus especificaciones
df_final = df_completado.with_columns(
    # Columnas que van obligatoriamente en 0 si no existía registro
    pl.col("cust_request_qty").fill_null(0),
    pl.col("cust_request_tn").fill_null(0.0),
    pl.col("tn").fill_null(0.0),
    
    # Columnas que mantienen su valor característico por combinación producto-cliente
    pl.col("plan_precios_cuidados").forward_fill().backward_fill().over(["customer_id", "product_id"]),
    pl.col("first_sell_in_period").forward_fill().backward_fill().over(["customer_id", "product_id"])
)

# 4. Si tienes más columnas descriptivas (ej: 'match'), las arrastramos dinámicamente
columnas_restantes = [col for col in df.columns if col not in ["customer_id", "product_id", "periodo", "cust_request_qty", "cust_request_tn", "tn", "plan_precios_cuidados", "first_sell_in_period"]]

if columnas_restantes:
    df_final = df_final.with_columns(
        pl.col(columnas_restantes).forward_fill().backward_fill().over(["customer_id", "product_id"])
    )

# Ordenamos el dataframe para su presentación final
df_final = df_final.sort(["product_id", "customer_id", "periodo"])

print(df_final)

shape: (10_096_720, 8)
┌────────────┬────────────┬─────────┬────────────┬────────────┬────────────┬───────────┬───────────┐
│ customer_i ┆ product_id ┆ periodo ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn        ┆ first_sel │
│ d          ┆ ---        ┆ ---     ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---       ┆ l_in_peri │
│ ---        ┆ i64        ┆ i64     ┆ s          ┆ ---        ┆ ---        ┆ f64       ┆ od        │
│ i64        ┆            ┆         ┆ ---        ┆ i64        ┆ f64        ┆           ┆ ---       │
│            ┆            ┆         ┆ i64        ┆            ┆            ┆           ┆ i64       │
╞════════════╪════════════╪═════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001      ┆ 201701  ┆ 0          ┆ 11         ┆ 99.43861   ┆ 99.43861  ┆ 201701    │
│ 10001      ┆ 20001      ┆ 201702  ┆ 0          ┆ 23         ┆ 198.84365  ┆ 198.84365 ┆ 201701    │
│ 10001      ┆ 20001      ┆ 201703  ┆ 0          ┆ 33         ┆ 92.4

## Aplanado de dataset

In [62]:
# Nos quedamos con los primeros 10 productos y los primeros 10 clientes para testear el pipeline
df_final = df_final.filter(
    (pl.col("product_id") <= 20010) & (pl.col("customer_id") <= 10010)
)

In [63]:
def achatar_dataset(df_final, granularidad, t0=201910, max_lags=None):
    """
    Achata el dataset (long -> wide): genera lags de tn + la clase.

    Params
    ------
    df_final    : pl.DataFrame ya completado (0s / nulls) con columnas
                  periodo, product_id, customer_id, tn.
    granularidad: "p" (producto) o "pc" (producto-cliente).
    t0          : periodo base en formato AAAAMM (default 201910). La clase es t0 + 2.
    max_lags    : cantidad de lags a generar -> columnas tn0..tn{max_lags}.
                  None = todo el historico disponible.

    Returns
    -------
    pl.DataFrame con keys + periodo (= t0) + tn0..tnN + clase_tn.
    Meses previos al first_sell del par quedan en NULL; meses sin venta dentro
    de su vida quedan en 0 (heredado del completado).
    """
    keys = ["product_id", "customer_id"] if granularidad == "pc" else ["product_id"]

    # lag = meses hacia atras respecto de t0 (escala lineal para cruzar anios)
    lin_t0 = (t0 // 100) * 12 + (t0 % 100)
    df_lag = df_final.with_columns(
        (lin_t0 - ((pl.col("periodo") // 100) * 12 + (pl.col("periodo") % 100))).alias("lag")
    )

    # Features: sumamos tn por (keys, lag).
    #   - granularidad "p": agrega tn sobre los clientes de cada producto.
    #   - granularidad "pc": es 1 fila por grupo (la suma no cambia nada).
    # Las combinaciones inexistentes (mes previo al first_sell) NO estan en el
    # long, y el pivot con "first" las deja en NULL (no en 0).
    cond = pl.col("lag") >= 0
    if max_lags is not None:
        cond = cond & (pl.col("lag") <= max_lags)

    feats_long = (
        df_lag
        .filter(cond)
        .group_by(keys + ["lag"])
        .agg(pl.col("tn").sum().alias("tn"))
    )

    features = (
        feats_long
        .with_columns(("tn" + pl.col("lag").cast(pl.Int64).cast(pl.String)).alias("lagname"))
        .pivot(values="tn", index=keys, on="lagname", aggregate_function="first")
    )

    # Garantizamos que existan TODAS las columnas tn0..tn{N}, con NULL donde falten
    # (el pivot solo crea columnas para los lags presentes en los datos).
    max_lag = max_lags if max_lags is not None else int(feats_long.select(pl.col("lag").max()).item() or 0)
    faltantes = [
        pl.lit(None, dtype=pl.Float64).alias(f"tn{k}")
        for k in range(max_lag + 1)
        if f"tn{k}" not in features.columns
    ]
    if faltantes:
        features = features.with_columns(faltantes)

    # Clase: tn dos meses adelante de t0 (lag == -2)
    clase = (
        df_lag
        .filter(pl.col("lag") == -2)
        .group_by(keys)
        .agg(pl.col("tn").sum().alias("clase_tn"))
    )

    df_achatado = (
        features
        .join(clase, on=keys, how="left")
        # Guardamos el periodo de referencia (= t0) para identificar / apilar distintos t0
        .with_columns(pl.lit(t0).cast(pl.Int64).alias("periodo"))
    )

    # Ordenamos columnas: keys, periodo, tn0, tn1, ..., clase_tn
    lag_cols = sorted(
        [c for c in df_achatado.columns if c.startswith("tn")],
        key=lambda c: int(c[2:]),
    )
    return df_achatado.select(keys + ["periodo"] + lag_cols + ["clase_tn"])


# --- Uso ---
# max_lags define cuantos lags generar (tn0..tn{max_lags}); None = todo el historico
df_achatado = achatar_dataset(df_final, granularidad, t0=201910, max_lags=12)
print(df_achatado)


shape: (96, 17)
┌────────────┬─────────────┬─────────┬───────────┬───┬──────────┬──────────┬───────────┬──────────┐
│ product_id ┆ customer_id ┆ periodo ┆ tn0       ┆ … ┆ tn10     ┆ tn11     ┆ tn12      ┆ clase_tn │
│ ---        ┆ ---         ┆ ---     ┆ ---       ┆   ┆ ---      ┆ ---      ┆ ---       ┆ ---      │
│ i64        ┆ i64         ┆ i64     ┆ f64       ┆   ┆ f64      ┆ f64      ┆ f64       ┆ f64      │
╞════════════╪═════════════╪═════════╪═══════════╪═══╪══════════╪══════════╪═══════════╪══════════╡
│ 20008      ┆ 10005       ┆ 201910  ┆ 0.78419   ┆ … ┆ 15.12969 ┆ 37.4844  ┆ 39.5233   ┆ 0.0      │
│ 20007      ┆ 10004       ┆ 201910  ┆ 5.72108   ┆ … ┆ 21.24971 ┆ 17.69837 ┆ 39.38591  ┆ 5.60432  │
│ 20005      ┆ 10010       ┆ 201910  ┆ 25.7712   ┆ … ┆ 0.0      ┆ 15.46272 ┆ 37.11053  ┆ 21.13238 │
│ 20005      ┆ 10009       ┆ 201910  ┆ 55.06446  ┆ … ┆ 45.31436 ┆ 41.21674 ┆ 68.07892  ┆ 27.57518 │
│ 20003      ┆ 10004       ┆ 201910  ┆ 222.21108 ┆ … ┆ 68.72775 ┆ 96.48469 ┆ 146.772

In [64]:
def normalizar_achatado(df_achatado, metodo="recta", norm_lags=None):
    """
    Normaliza fila a fila las columnas tn* y clase_tn de df_achatado.

    Los parametros de normalizacion (B0, B1) se calculan SOLO con la ventana
    tn0..tn{norm_lags} (nunca con clase_tn -> evita leakage) y luego se aplican
    a todos los tn* y a clase_tn, generando tn0_norm..tnN_norm y clase_tn_norm.

    Params
    ------
    metodo:
      "recta"   -> ajuste lineal por minimos cuadrados sobre la ventana.
                   B0 = ordenada al origen, B1 = pendiente.
                   norm = valor - (B0 + B1 * lag)   (clase_tn esta en lag = -2)
      "zscore"  -> B0 = media, B1 = desvio.   norm = (valor - B0) / B1
      "minmax"  -> B0 = min,   B1 = max-min.  norm = (valor - B0) / B1
      "media"   -> B0 = 0,     B1 = media.    norm = valor / B1
    norm_lags:
      hasta que lag usar para calcular B0/B1 (ventana tn0..tn{norm_lags}).
      None = usa todos los lags disponibles.

    Devuelve df_achatado + columnas B0, B1, tn*_norm y clase_tn_norm.
    """
    lag_cols = sorted(
        [c for c in df_achatado.columns if c.startswith("tn") and c[2:].isdigit()],
        key=lambda c: int(c[2:]),
    )
    n_max = int(lag_cols[-1][2:])
    if norm_lags is None:
        norm_lags = n_max
    fit_cols = [f"tn{k}" for k in range(norm_lags + 1)]

    if metodo == "recta":
        # Minimos cuadrados sobre la ventana, ignorando los nulls
        n   = pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Float64) for c in fit_cols])
        Sy  = pl.sum_horizontal([pl.col(c) for c in fit_cols])
        Sx  = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxx = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k * k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxy = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(pl.col(c) * float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        denom = n * Sxx - Sx * Sx
        slope = pl.when(denom != 0).then((n * Sxy - Sx * Sy) / denom).otherwise(0.0)
        inter = pl.when(n > 0).then((Sy - slope * Sx) / n).otherwise(None)

        df = df_achatado.with_columns(inter.alias("B0"), slope.alias("B1"))

        # norm = valor - (B0 + B1 * lag);  clase_tn esta en lag = -2 (t0 + 2)
        exprs = [
            (pl.col(c) - (pl.col("B0") + pl.col("B1") * float(int(c[2:])))).alias(f"{c}_norm")
            for c in lag_cols
        ]
        exprs.append((pl.col("clase_tn") - (pl.col("B0") + pl.col("B1") * (-2.0))).alias("clase_tn_norm"))
        return df.with_columns(exprs)

    # --- metodos afines: norm = (valor - B0) / B1 ---
    if metodo == "zscore":
        B0 = pl.mean_horizontal(fit_cols)
        B1 = pl.concat_list(fit_cols).list.std()
    elif metodo == "minmax":
        B0 = pl.min_horizontal(fit_cols)
        B1 = pl.max_horizontal(fit_cols) - pl.min_horizontal(fit_cols)
    elif metodo == "media":
        B0 = pl.lit(0.0)
        B1 = pl.mean_horizontal(fit_cols)
    else:
        raise ValueError(f"metodo no soportado: {metodo}")

    df = df_achatado.with_columns(B0.alias("B0"), B1.alias("B1"))
    # B1 seguro para no dividir por 0 ni por null (series constantes / sin datos)
    b1_safe = pl.when((pl.col("B1") == 0) | pl.col("B1").is_null()).then(1.0).otherwise(pl.col("B1"))
    exprs = [((pl.col(c) - pl.col("B0")) / b1_safe).alias(f"{c}_norm") for c in lag_cols]
    exprs.append(((pl.col("clase_tn") - pl.col("B0")) / b1_safe).alias("clase_tn_norm"))
    return df.with_columns(exprs)


# --- Uso ---
# metodo: "recta" | "zscore" | "minmax" | "media"
# norm_lags: ventana tn0..tn{norm_lags} para calcular B0/B1 (None = todos los lags)
df_norm = normalizar_achatado(df_achatado, metodo="media", norm_lags=None)
print(df_norm)


shape: (96, 33)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0       ┆ … ┆ tn10_norm ┆ tn11_norm ┆ tn12_norm ┆ clase_tn_ │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ norm      │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆           ┆           ┆           ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20008      ┆ 10005     ┆ 201910  ┆ 0.78419   ┆ … ┆ 0.514524  ┆ 1.274754  ┆ 1.344092  ┆ 0.0       │
│ 20007      ┆ 10004     ┆ 201910  ┆ 5.72108   ┆ … ┆ 0.744962  ┆ 0.620461  ┆ 1.380772  ┆ 0.196474  │
│ 20005      ┆ 10010     ┆ 201910  ┆ 25.7712   ┆ … ┆ 0.0       ┆ 1.227702  ┆ 2.946485  ┆ 1.677859  │
│ 20005      ┆ 10009     ┆ 201910  ┆ 55.06446  ┆ … ┆ 1.155765  ┆ 1.051253  

In [65]:
def desnormalizar(df, col_norm, metodo, lag=-2, alias="pred_tn"):
    """
    Recupera la escala original (toneladas) de una columna normalizada,
    usando los parametros B0/B1 guardados por normalizar_achatado.
    Es el inverso exacto de normalizar_achatado.

    Params
    ------
    df       : DataFrame que tiene las columnas B0, B1 y col_norm.
    col_norm : columna normalizada a desnormalizar (ej. la prediccion del modelo).
    metodo   : el MISMO metodo usado al normalizar ("recta"|"zscore"|"minmax"|"media").
    lag      : posicion temporal de la columna respecto de t0.
               clase_tn (y su prediccion) esta en lag = -2 (t0 + 2); tn{k} esta en lag = k.
    alias    : nombre de la columna de salida.

    Devuelve df + columna 'alias' en escala original.
    """
    if metodo == "recta":
        # inverso de: norm = valor - (B0 + B1 * lag)
        valor = pl.col(col_norm) + (pl.col("B0") + pl.col("B1") * float(lag))
    elif metodo in ("zscore", "minmax", "media"):
        # inverso de: norm = (valor - B0) / B1  (con el mismo B1 seguro del forward)
        b1_safe = pl.when((pl.col("B1") == 0) | pl.col("B1").is_null()).then(1.0).otherwise(pl.col("B1"))
        valor = pl.col(col_norm) * b1_safe + pl.col("B0")
    else:
        raise ValueError(f"metodo no soportado: {metodo}")
    return df.with_columns(valor.alias(alias))


# --- Uso ---
# Chequeo round-trip: desnormalizar clase_tn_norm (lag=-2) tiene que devolver clase_tn.
METODO = "media"
df_check = desnormalizar(df_norm, "clase_tn_norm", METODO, lag=-2, alias="clase_tn_rec")
error_max = df_check.select((pl.col("clase_tn_rec") - pl.col("clase_tn")).abs().max()).item()
print("Error maximo de round-trip:", error_max)

# Cuando tengas la prediccion del modelo en escala normalizada (ej. columna 'clase_tn_norm_pred'),
# la pasas a toneladas asi:
# df_pred = desnormalizar(df_pred, "clase_tn_norm_pred", METODO, lag=-2, alias="tn_pred")


Error maximo de round-trip: 2.842170943040401e-14


In [66]:
def agregar_deltas(df_norm, salto=2):
    """
    Agrega columnas de delta (saltos de 'salto' periodos) sobre las columnas
    NORMALIZADAS. No elimina ninguna columna, solo agrega.

      tn{k}_delta    = tn{k}_norm - tn{k+salto}_norm   (k = 0 .. maxlag-salto)
      clase_tn_delta = clase_tn_norm - tn0_norm        (cambio a 2 periodos -> target)

    tn0_delta es el delta t0 - t2 (el cambio en el ti mas reciente).
    Para este problema salto=2 porque se predice a 2 periodos.
    """
    norm_lags = sorted(
        int(c[2:-5]) for c in df_norm.columns
        if c.startswith("tn") and c.endswith("_norm") and c[2:-5].isdigit()
    )
    max_k = max(norm_lags)
    exprs = [
        (pl.col(f"tn{k}_norm") - pl.col(f"tn{k + salto}_norm")).alias(f"tn{k}_delta")
        for k in range(0, max_k - salto + 1)
    ]
    # Delta de la clase: valor de la clase normalizada menos t0 normalizado
    exprs.append((pl.col("clase_tn_norm") - pl.col("tn0_norm")).alias("clase_tn_delta"))
    return df_norm.with_columns(exprs)


# --- Uso ---
df_norm = agregar_deltas(df_norm, salto=2)
print(df_norm)


shape: (96, 45)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0       ┆ … ┆ tn8_delta ┆ tn9_delta ┆ tn10_delt ┆ clase_tn_ │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ a         ┆ delta     │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆           ┆           ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20008      ┆ 10005     ┆ 201910  ┆ 0.78419   ┆ … ┆ 0.738895  ┆ 1.51477   ┆ -0.829568 ┆ -0.026668 │
│ 20007      ┆ 10004     ┆ 201910  ┆ 5.72108   ┆ … ┆ -0.088003 ┆ 0.398746  ┆ -0.63581  ┆ -0.004093 │
│ 20005      ┆ 10010     ┆ 201910  ┆ 25.7712   ┆ … ┆ 0.0       ┆ -1.227702 ┆ -2.946485 ┆ -0.368311 │
│ 20005      ┆ 10009     ┆ 201910  ┆ 55.06446  ┆ … ┆ -0.383429 ┆ -0.215378 

In [74]:
df_norm.columns

['product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'clase_tn_delta']

In [75]:
#checkeo
df_norm['product_id', 'customer_id', 'periodo', 'tn0_norm', 'tn1_norm','tn2_norm','tn3_norm','tn0_delta',
 'tn1_delta','clase_tn_norm','clase_tn_delta']

product_id,customer_id,periodo,tn0_norm,tn1_norm,tn2_norm,tn3_norm,tn0_delta,tn1_delta,clase_tn_norm,clase_tn_delta
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
20008,10005,201910,0.026668,1.056072,0.053337,0.085339,-0.026669,0.970733,0.0,-0.026668
20007,10004,201910,0.200567,2.435453,0.687657,0.315177,-0.48709,2.120276,0.196474,-0.004093
20005,10010,201910,2.04617,1.227702,1.636936,0.0,0.409234,1.227702,1.677859,-0.368311
20005,10009,201910,1.404446,1.596708,0.364806,1.288979,1.03964,0.307729,0.703318,-0.701128
20003,10004,201910,2.527821,0.523974,0.375279,0.939672,2.152542,-0.415698,0.431925,-2.095896
…,…,…,…,…,…,…,…,…,…,…
20007,10009,201910,0.918314,1.209689,1.459636,0.30104,-0.541321,0.908648,0.364562,-0.553752
20006,10006,201910,1.96586,1.076124,0.337607,1.076124,1.628253,0.0,0.803928,-1.161933
20003,10007,201910,0.28196,1.429587,1.113,0.16324,-0.83104,1.266347,1.855001,1.573041
